# Player Similarity Search and PCA

This notebook compares football player similarity using multiple distance metrics, then visualizes the top matches with a custom PCA implementation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load data
# NOTE: update the path here if your CSV is stored somewhere else.
df = pd.read_csv('data.csv')

features = [
    'Crossing', 'Finishing', 'HeadingAccuracy', 'ShortPassing', 'Volleys',
    'Dribbling', 'Curve', 'FKAccuracy', 'LongPassing', 'BallControl',
    'Acceleration', 'SprintSpeed', 'Agility', 'Reactions', 'Balance',
    'ShotPower', 'Jumping', 'Stamina', 'Strength', 'LongShots',
    'Aggression', 'Interceptions', 'Positioning', 'Vision', 'Penalties',
    'Composure', 'Marking', 'StandingTackle', 'SlidingTackle'
]

df = df.dropna(subset=features + ['Name']).reset_index(drop=True)

target_name = 'M. Salah'
target_idx = df[df['Name'] == target_name].index[0]

X_raw = df[features].values
target_vector_raw = X_raw[target_idx]

## Standardization and Similarity Metrics

This section standardizes the player features and evaluates three similarity measures: Euclidean distance, Manhattan distance, and cosine similarity.

In [ ]:
def standardize_features(X):
    """Calculate z-scores for the feature matrix."""
    means = np.mean(X, axis=0)
    stds = np.std(X, axis=0)
    return (X - means) / (stds + 1e-8)

X_std = standardize_features(X_raw)
target_vector_std = X_std[target_idx]


def euclidean_distance(target, data):
    """Lower score means more similar."""
    return np.sqrt(np.sum((data - target) ** 2, axis=1))


def manhattan_distance(target, data):
    """Lower score means more similar."""
    return np.sum(np.abs(data - target), axis=1)


def cosine_similarity(target, data):
    """Higher score means more similar."""
    dot_product = np.dot(data, target)
    norm_data = np.linalg.norm(data, axis=1)
    norm_target = np.linalg.norm(target)
    return dot_product / (norm_data * norm_target + 1e-8)


def get_top_5(metrics_array, df, target_idx, ascending=True):
    """Return the top 5 similar players, excluding the target."""
    if ascending:
        best_indices = np.argsort(metrics_array)
    else:
        best_indices = np.argsort(metrics_array)[::-1]

    best_indices = [i for i in best_indices if i != target_idx][:5]

    return pd.DataFrame({
        'Name': df.iloc[best_indices]['Name'],
        'Overall': df.iloc[best_indices]['Overall'],
        'Metric_Score': metrics_array[best_indices],
    })

print("=== EXPERIMENT 1: RAW UNSTANDARDIZED DATA ===")
print("\nEuclidean Top 5:")
euc_raw = euclidean_distance(target_vector_raw, X_raw)
print(get_top_5(euc_raw, df, target_idx, ascending=True))

print("\n=== EXPERIMENT 2: STANDARDIZED DATA ===")
print("\nEuclidean Top 5:")
euc_std = euclidean_distance(target_vector_std, X_std)
print(get_top_5(euc_std, df, target_idx, ascending=True))

print("\nManhattan Top 5 (Standardized):")
man_std = manhattan_distance(target_vector_std, X_std)
print(get_top_5(man_std, df, target_idx, ascending=True))

print("\nCosine Similarity Top 5 (Standardized):")
cos_std = cosine_similarity(target_vector_std, X_std)
print(get_top_5(cos_std, df, target_idx, ascending=False))

## PCA Visualization

Here the standardized feature space is projected to two dimensions using a custom PCA implementation, then the target player and closest matches are highlighted.

In [ ]:
def custom_pca(X, n_components=2):
    """PCA implemented from scratch."""
    X_centered = X - np.mean(X, axis=0)
    cov_matrix = np.cov(X_centered, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    sorted_indices = np.argsort(eigenvalues)[::-1]
    top_eigenvectors = eigenvectors[:, sorted_indices[:n_components]]
    return np.dot(X_centered, top_eigenvectors)

X_pca = custom_pca(X_std, n_components=2)

best_indices_euc = np.argsort(euc_std)
shortlist_idx = [i for i in best_indices_euc if i != target_idx][:5]

plt.figure(figsize=(10, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c='lightgray', alpha=0.5, label='Player Pool', s=10)
plt.scatter(X_pca[shortlist_idx, 0], X_pca[shortlist_idx, 1], c='blue', s=100, edgecolor='k', label='Shortlist (Top 5 Euclidean)')
plt.scatter(X_pca[target_idx, 0], X_pca[target_idx, 1], c='red', marker='*', s=300, edgecolor='k', label='Target (M. Salah)')

plt.title('Player Pool PCA Projection (2D)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.show()